# Praktikum Machine Learning: Implementasi Word2Vec
Notebook ini dirancang untuk dijalankan di **Google Colab** guna memahami dan mempraktikkan konsep **Word Embedding** menggunakan algoritma **Word2Vec (Gensim)**.

---

### Apa itu Word2Vec?
**Word2Vec** adalah teknik *Natural Language Processing* (NLP) yang diperkenalkan oleh Tomas Mikolov dkk. (Google, 2013) untuk merepresentasikan kata ke dalam bentuk vektor numerik padat (*dense vector*) berdimensi rendah ($D = 50, 100, 300$).

Berbeda dengan metode tradisional seperti **One-Hot Encoding** atau **TF-IDF** yang menghasilkan vektor renggang (*sparse*) tanpa memahami makna kata, Word2Vec mampu menangkap **hubungan semantik (makna)** dan **konteks sintaksis** antar kata berdasarkan prinsip distribusi lingustik:
> *"A word is characterized by the company it keeps"* (Sebuah kata dicirikan oleh lingkungan kata di sekitarnya).

### Fitur Utama Word2Vec:
1. **Semantic Similarity**: Kata dengan makna serupa berada di posisi koordinat yang saling berdekatan (misal: `kucing` dekat dengan `anjing`).
2. **Vector Arithmetic (Analogi Kata)**: Hubungan antar kata dapat dihitung secara matematis:
   $$\vec{v}(\text{Raja}) - \vec{v}(\text{Pria}) + \vec{v}(\text{Wanita}) \approx \vec{v}(\text{Ratu})$$
3. **Dua Arsitektur Utama**:
   - **CBOW (Continuous Bag of Words)**: Memprediksi kata target berdasarkan kata-kata di sekitarnya (konteks).
   - **Skip-Gram**: Memprediksi kata-kata konteks di sekitarnya berdasarkan satu kata target.

## 1. Install & Import Library
Di Google Colab, library `gensim`, `matplotlib`, `scikit-learn`, `pandas`, dan `numpy` biasanya sudah terpasang. Kita juga menginstall `PySastrawi` untuk kebutuhan preprocessing bahasa Indonesia.

In [ ]:
# Install library pendukung jika belum tersedia
!pip install -q gensim PySastrawi

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from sklearn.decomposition import PCA

print("Semua library berhasil diimpor dengan sukses!")

## 2. Menyiapkan Dataset Korpus Kalimat
Kita menyiapkan korpus kalimat bahasa Indonesia dengan beragam domain semantik (kerajaan, keluarga, teknologi, dan hewan) yang dirancang agar pola hubungan antar kata dapat dipelajari oleh model.

In [ ]:
# Kumpulan kalimat contoh (Korpus teks)
korpus_mentah = [
    # Hubungan Kerajaan & Gender
    "raja adalah pemimpin pria di sebuah kerajaan besar",
    "ratu adalah pemimpin wanita di sebuah kerajaan besar",
    "pangeran adalah putra dari raja dan ratu istana",
    "putri adalah putri dari raja dan ratu istana",
    "seorang raja dan ratu memerintah istana megah kerajaan",
    "pria dan wanita memiliki hak yang sama sebagai manusia",
    "pria adalah laki laki dewasa sedangkan wanita adalah perempuan dewasa",
    "ayah adalah pria kepala keluarga dan ibu adalah wanita pengatur rumah tangga",
    "kakek dan nenek merawat cucu mereka dengan penuh kasih sayang",
    "ayah dan ibu menyayangi anak laki laki dan anak perempuan mereka",

    # Hubungan Teknologi & Pemrograman
    "python adalah bahasa pemrograman populer untuk data science dan machine learning",
    "programmer menulis kode menggunakan bahasa python untuk membangun aplikasi",
    "machine learning dan deep learning adalah cabang utama dari kecerdasan buatan",
    "komputer dan laptop digunakan oleh programmer untuk komputasi data",
    "algoritma kecerdasan buatan memproses data besar dalam sistem komputer modern",
    "data science membutuhkan pemahaman statistik matematika dan pemrograman python",
    "model machine learning belajar mengenali pola data secara otomatis",

    # Hubungan Hewan & Peliharaan
    "kucing adalah hewan mamalia lucu pemakan daging",
    "anjing adalah hewan mamalia setia yang sering dipelihara manusia",
    "kucing dan anjing merupakan hewan peliharaan populer di rumah",
    "singa dan harimau adalah hewan buas pemakan daging di hutan",
    "hewan mamalia seperti kucing anjing singa dan harimau melahirkan anak"
]

print(f"Total kalimat dalam korpus: {len(korpus_mentah)}")
for i, kal in enumerate(korpus_mentah[:5], 1):
    print(f"{i}. {kal}")

## 3. Preprocessing dan Tokenisasi Teks
Word2Vec membutuhkan input berupa **list of sentences**, di mana setiap kalimat adalah **list of tokens (kata-kata)**:
Contoh format yang diharapkan: `[['raja', 'pemimpin', 'kerajaan'], ['ratu', 'istana'], ...]`

In [ ]:
def bersihkan_dan_tokenisasi(kalimat):
    # 1. Ubah ke huruf kecil (case folding)
    kalimat = kalimat.lower()
    # 2. Hapus tanda baca dan karakter selain alfabet dan spasi
    kalimat = re.sub(r'[^a-z\s]', ' ', kalimat)
    # 3. Tokenisasi berdasarkan spasi
    tokens = kalimat.split()
    return tokens

# Terapkan ke seluruh korpus
korpus_token = [bersihkan_dan_tokenisasi(k) for k in korpus_mentah]

# Tampilkan contoh hasil tokenisasi
print("Contoh token kalimat pertama:")
print(korpus_token[0])

# Hitung total token unik
semua_kata = [kata for kalimat in korpus_token for kata in kalimat]
print(f"\nTotal kata dalam seluruh korpus: {len(semua_kata)}")
print(f"Total kata unik (vocabulary): {len(set(semua_kata))}")

## 4. Melatih Model Word2Vec (Gensim)

### Parameter Kunci `Word2Vec`:
- `sentences`: Data masukan berupa list token kalimat.
- `vector_size`: Dimensi dari vektor kata (misal: 50 atau 100 dimensi). Semakin besar, semakin kaya representasinya, namun membutuhkan lebih banyak data.
- `window`: Jarak konteks (jumlah kata ke kiri dan ke kanan yang diperhatikan).
- `min_count`: Frekuensi minimum kemunculan kata agar dimasukkan ke dalam kosakata (*vocabulary*).
- `sg`: Arsitektur pelatihan (`0` untuk **CBOW**, `1` untuk **Skip-Gram**).
- `epochs`: Berapa kali model mengulang seluruh dataset selama pelatihan.
- `workers`: Jumlah thread CPU untuk mempercepat proses pelatihan.
- `seed`: Nilai random seed untuk hasil yang konsisten (*reproducible*).

In [ ]:
# Inisialisasi dan training model Word2Vec (Arsitektur Skip-Gram: sg=1)
model_w2v = Word2Vec(
    sentences=korpus_token,
    vector_size=50,       # 50 dimensi fitur per kata
    window=4,             # Jendela konteks: 4 kata sebelum & sesudah
    min_count=1,          # Sertakan semua kata meski hanya muncul 1 kali
    sg=1,                 # 1 = Skip-Gram (bagus untuk korpus kecil-menengah)
    epochs=200,           # Iterasi training agar konvergen pada dataset ringkas
    seed=42,
    workers=4
)

print("Training Word2Vec berhasil!")
print(f"Jumlah kosakata yang dipelajari model: {len(model_w2v.wv.index_to_key)}")

## 5. Eksplorasi Representasi Vektor Kata
Setiap kata kini telah dipetakan ke dalam array angka berukuran 50 dimensi (*dense vector*).

In [ ]:
# Melihat representasi vektor dari kata 'raja'
vektor_raja = model_w2v.wv['raja']

print(f"Tipe data vektor: {type(vektor_raja)}")
print(f"Bentuk (shape) vektor: {vektor_raja.shape} (berarti {vektor_raja.shape[0]} dimensi)")
print("\n10 nilai pertama dalam vektor 'raja':")
print(vektor_raja[:10])

## 6. Analisis Kemiripan Semantik (Cosine Similarity)
Menggunakan fungsi bawaan `model.wv.similarity(kata1, kata2)` untuk mengukur seberapa mirip makna dua kata (nilai berkisar antara -1 hingga 1, di mana 1 artinya sangat mirip/identik dalam konteks).

In [ ]:
# Bandingkan kemiripan antar kata terkait vs kata tidak terkait
pasangan_uji = [
    ("raja", "ratu"),
    ("pria", "wanita"),
    ("ayah", "ibu"),
    ("kucing", "anjing"),
    ("python", "programmer"),
    ("raja", "komputer"),     # Kata dari domain berbeda
    ("kucing", "algoritma")    # Kata dari domain berbeda
]

hasil_similarity = []
for kata1, kata2 in pasangan_uji:
    skor = model_w2v.wv.similarity(kata1, kata2)
    hasil_similarity.append({
        'Kata 1': kata1,
        'Kata 2': kata2,
        'Cosine Similarity': round(skor, 4)
    })

df_sim = pd.DataFrame(hasil_similarity)
df_sim

## 7. Mencari Kata Paling Mirip (`most_similar`)
Fungsi `model.wv.most_similar(kata, topn=k)` mencari $k$ kata dalam seluruh kosakata yang memiliki jarak kosinus terdekat dengan kata masukan.

In [ ]:
def tampilkan_kata_terdekat(kata, topn=5):
    if kata not in model_w2v.wv:
        print(f"Kata '{kata}' tidak ditemukan dalam kosakata model.")
        return
    
    print(f"Kata yang paling mirip/relevan dengan '{kata}':")
    mirip = model_w2v.wv.most_similar(kata, topn=topn)
    for rank, (k, skor) in enumerate(mirip, 1):
        print(f"  {rank}. {k:<15} (skor: {skor:.4f})")
    print()

tampilkan_kata_terdekat("raja")
tampilkan_kata_terdekat("python")
tampilkan_kata_terdekat("kucing")

## 8. Analogi Kata & Aritmatika Vektor (Word Vector Arithmetic)
Salah satu keajaiban Word2Vec adalah kemampuan operasi vektor:

$$\text{Target} = \text{positif}_1 + \text{positif}_2 - \text{negatif}_1$$

Contoh klasik: **Raja - Pria + Wanita = Ratu**  
Di Gensim, kita menggunakan parameter `positive=['raja', 'wanita']` dan `negative=['pria']`.

In [ ]:
print("=== ANALOGI 1: Raja - Pria + Wanita ===")
hasil_analogi = model_w2v.wv.most_similar(
    positive=["raja", "wanita"],
    negative=["pria"],
    topn=3
)
for kata, skor in hasil_analogi:
    print(f"  -> {kata:<15} (skor: {skor:.4f})")

print("\n=== ANALOGI 2: Ayah - Pria + Wanita ===")
hasil_analogi_2 = model_w2v.wv.most_similar(
    positive=["ayah", "wanita"],
    negative=["pria"],
    topn=3
)
for kata, skor in hasil_analogi_2:
    print(f"  -> {kata:<15} (skor: {skor:.4f})")

## 9. Menemukan Kata yang Tidak Cocok (`doesnt_match`)
Word2Vec dapat mendeteksi kata yang ganjil (*outlier*) dari sekelompok kata berdasarkan rata-rata jarak semantiknya.

In [ ]:
daftar_kata_1 = ["kucing", "anjing", "singa", "komputer"]
outlier_1 = model_w2v.wv.doesnt_match(daftar_kata_1)
print(f"Dari {daftar_kata_1} -> Kata yang tidak cocok: '{outlier_1}'")

daftar_kata_2 = ["raja", "ratu", "pangeran", "python"]
outlier_2 = model_w2v.wv.doesnt_match(daftar_kata_2)
print(f"Dari {daftar_kata_2} -> Kata yang tidak cocok: '{outlier_2}'")

## 10. Visualisasi 2D Vektor Kata Menggunakan PCA
Karena ruang vektor berdimensi 50 ($D=50$) tidak bisa divisualisasikan langsung oleh mata manusia, kita memproyeksikannya ke dalam 2 dimensi menggunakan **Principal Component Analysis (PCA)**.

In [ ]:
# Pilih kata-kata representatif dari beberapa kluster semantik
kata_pilihan = [
    # Kluster Kerajaan & Gender
    "raja", "ratu", "pangeran", "putri", "pria", "wanita", "istana", "ayah", "ibu",
    # Kluster Teknologi
    "python", "programmer", "komputer", "laptop", "algoritma", "data",
    # Kluster Hewan
    "kucing", "anjing", "singa", "harimau", "mamalia"
]

# Ambil vektor dari kata-kata yang dipilih
vektor_pilihan = np.array([model_w2v.wv[kata] for kata in kata_pilihan])

# Reduksi dimensi 50D -> 2D dengan PCA
pca = PCA(n_components=2, random_state=42)
koordinat_2d = pca.fit_transform(vektor_pilihan)

# Visualisasi Scatter Plot
plt.figure(figsize=(12, 8))
plt.scatter(koordinat_2d[:, 0], koordinat_2d[:, 1], color='#3498db', edgecolors='black', s=120, alpha=0.8)

# Tambahkan label nama kata di samping setiap titik koordinat
for i, kata in enumerate(kata_pilihan):
    plt.annotate(
        kata,
        xy=(koordinat_2d[i, 0], koordinat_2d[i, 1]),
        xytext=(6, 4),
        textcoords='offset points',
        fontsize=11,
        fontweight='semibold',
        color='#2c3e50'
    )

plt.title("Visualisasi 2D Word Embedding (PCA) - Word2Vec", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Principal Component 1", fontsize=11)
plt.ylabel("Principal Component 2", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 11. Menyimpan dan Memuat Kembali Model
Model yang sudah dilatih dapat disimpan ke dalam media penyimpanan (disk / Google Drive) sehingga tidak perlu dilatih ulang di masa mendatang.

In [ ]:
# 1. Simpan seluruh model (bisa dilatih lebih lanjut / re-training)
model_w2v.save("model_word2vec_praktikum.model")
print("Model berhasil disimpan sebagai 'model_word2vec_praktikum.model'")

# 2. Memuat kembali model
model_termuat = Word2Vec.load("model_word2vec_praktikum.model")
print("Model berhasil dimuat kembali!")

# 3. Uji coba model yang dimuat
print("Skor kemiripan 'raja' dan 'ratu' dari model termuat:", 
      model_termuat.wv.similarity('raja', 'ratu'))

## 12. Bonus: Menggunakan Pre-trained Model (Gensim Downloader)
Dalam proyek industri nyata, kita sering menggunakan model Word2Vec atau GloVe yang sudah dilatih sebelumnya (*pre-trained*) menggunakan miliaran kata dari Wikipedia atau Google News.

> **Catatan**: Unduhan model ini membutuhkan koneksi internet dan kuota (sekitar 66 MB untuk `glove-wiki-gigaword-50`).

In [ ]:
import gensim.downloader as api

print("Mengunduh model pre-trained kecil (GloVe 50d berbasis Wikipedia)...")
pretrained_model = api.load("glove-wiki-gigaword-50")

# Uji coba analogi kata bahasa Inggris pada model pre-trained riil:
# king - man + woman = queen
hasil_pretrained = pretrained_model.most_similar(positive=["king", "woman"], negative=["man"], topn=3)

print("\nHasil Analogi (king - man + woman) pada model Wikipedia:")
for kata, skor in hasil_pretrained:
    print(f"  -> {kata}: {skor:.4f}")

### Rangkuman & Poin Penting:
1. **Dense vs Sparse**: Word2Vec menghasilkan representasi padat (*dense vector*), berbeda dengan One-Hot atau TF-IDF yang menghasilkan matriks jarang (*sparse*).
2. **Makna Semantik**: Kata yang muncul dalam konteks yang mirip akan memiliki vektor yang berdekatan dalam ruang multi-dimensi.
3. **Arsitektur**: Gunakan **CBOW** untuk kecepatan pada dataset besar, dan **Skip-Gram** untuk menangkap detail kata langka dan dataset berukuran kecil hingga menengah.
4. **Aritmatika Vektor**: Mampu menyelesaikan permasalahan analogi kata seperti `king - man + woman = queen`.